In [1]:
import pandas as pd
import dotenv
from datetime import datetime, timedelta, timezone

dotenv.load_dotenv()

True

In [4]:
with open('data/BlackMentalHealth_submissions.jsonl', 'r', encoding='utf-8') as f:
    for i in range(5):
        print(f.readline())

{"all_awardings":[],"allow_live_comments":false,"archived":false,"author":"MsRawrie","author_created_utc":1390475446,"author_flair_background_color":"transparent","author_flair_css_class":null,"author_flair_richtext":[],"author_flair_template_id":"e562342e-6301-11ea-9491-0eb4e2bcd141","author_flair_text":"Black Mental Health Matters","author_flair_text_color":"light","author_flair_type":"text","author_fullname":"t2_exw9o","author_patreon_flair":false,"author_premium":false,"awarders":[],"can_gild":true,"can_mod_post":false,"category":null,"content_categories":null,"contest_mode":false,"created_utc":1582686915,"discussion_type":null,"distinguished":null,"domain":"self.BlackMentalHealth","edited":1582688659,"gilded":0,"gildings":{},"hidden":false,"id":"f9mog4","is_crosspostable":true,"is_meta":false,"is_original_content":false,"is_reddit_media_domain":false,"is_robot_indexable":true,"is_self":true,"is_video":false,"link_flair_background_color":"","link_flair_css_class":null,"link_flair_r

In [ ]:
file_paths = [
    "data/malementalhealth_submissions.jsonl", 
    "data/MentalHealthPH_submissions.jsonl", 
    "data/MentalHealthSupport_submissions.jsonl", 
    "data/MentalHealthUK_submissions.jsonl", 
    "data/BlackMentalHealth_submissions.jsonl"]

chunksize = 10_00_000  # tune for your RAM

cutoff_dt = datetime.now(timezone.utc) - timedelta(days=365*2)
cutoff_ts = cutoff_dt.timestamp()

frames = []

for file in file_paths:
    print("processing file: ", file)
    for chunk in pd.read_json(file, lines=True, chunksize=chunksize):
        # keep only last 2 years
        chunk = chunk[chunk["created_utc"] >= cutoff_ts]

        # drop deleted/removed posts
        chunk = chunk[~chunk["selftext"].isin(["[deleted]", "[removed]"])]
        chunk = chunk[~chunk["author"].isin(["[deleted]", "[removed]"])]

        if len(chunk):
            frames.append(chunk)

time_filtered_df = pd.concat(frames, ignore_index=True)
clean_df = time_filtered_df[
    [
        "author",
        "selftext",
        "created_utc",
        "title",
        "permalink",
        "score",
        "id",
        "subreddit",
        "subreddit_id",
        "link_flair_text",
        "retrieved_on",
    ]
].dropna()

# add t3_ prefix to id and rename to post_id
clean_df["post_id"] = "t3_" + clean_df["id"].astype(str)
clean_df = clean_df.drop(columns=["id"])

print(len(clean_df), "rows in last 1 years after filtering")

60346 rows in last 1 years after filtering


## Inspecting Data

Huh. Turns out there's a lot of entries with no text. Let's get rid of them. Only keep entries with 100+ words.

In [4]:
selftext_lengths = clean_df["selftext"].fillna("").apply(lambda x: len(str(x).split()))
title_lengths = clean_df["title"].fillna("").apply(lambda x: len(str(x).split()))

zero_len_df = clean_df[selftext_lengths == 0]

print(f"Number of posts with 0-word selftext: {len(zero_len_df)}")
print(zero_len_df[["author", "title", "selftext", "permalink"]].head(5).to_string(index=False))

Number of posts with 0-word selftext: 1008
              author                                                                                                          title selftext                                                                            permalink
    HungryAnswer1776 It feels like I'm living a lie fabricated by the people around me and I don't know how to know that's not true             /r/mentalhealth/comments/1cmexbx/it_feels_like_im_living_a_lie_fabricated_by_the/
  Beautiful-Money343                                                     I AM SICK OF RE-STARTING MY WEIGHTLOSS JOURNEY !!!!! DAY-3          /r/mentalhealth/comments/1ev3bu9/i_am_sick_of_restarting_my_weightloss_journey_day3/
     Glass_Sort_5661                  Your mental health is a priority. Your happiness is essential. Your self-care is a necessity.             /r/mentalhealth/comments/1evx05r/your_mental_health_is_a_priority_your_happiness/
Subject-Channel-8959                                 

In [5]:
clean_df = clean_df[selftext_lengths > 100]
print(f"{len(clean_df)} rows remain after dropping posts with 100-word selftext")

138506 rows remain after dropping posts with 100-word selftext


In [6]:
max_title_len = clean_df["title"].fillna("").apply(lambda x: len(str(x))).max()
print(f"Maximum title length in characters: {max_title_len}")

Maximum title length in characters: 300


In [7]:
# ChromaDB limits documents to 16KB (~16k chars). Truncate selftext so title + " | " + selftext <= 16000
MAX_DOC_CHARS = 16000
SEPARATOR = " | "

before_len = clean_df["title"].fillna("").str.len() + len(SEPARATOR) + clean_df["selftext"].fillna("").str.len()
n_truncated = (before_len > MAX_DOC_CHARS).sum()

def truncate_selftext(row):
    title = str(row["title"]) if pd.notna(row["title"]) else ""
    selftext = str(row["selftext"]) if pd.notna(row["selftext"]) else ""
    max_selftext_len = MAX_DOC_CHARS - len(title) - len(SEPARATOR)
    if len(selftext) > max_selftext_len:
        return selftext[:max_selftext_len]
    return selftext

clean_df["selftext"] = clean_df.apply(truncate_selftext, axis=1)
print(f"Documents truncated to {MAX_DOC_CHARS} chars. Rows affected: {n_truncated}")

# Show an example of a truncated document
if n_truncated > 0:
    example_idx = before_len[before_len > MAX_DOC_CHARS].index[0]
    row = clean_df.loc[example_idx]
    doc = row["title"] + SEPARATOR + row["selftext"]
    print(f"\nExample truncated doc (post_id={row['post_id']}, orig length={int(before_len.loc[example_idx])} chars):")
    print(f"  Total length now: {len(doc)} chars")
    print(f"  Title: {row['title'][:80]}...")
    print(f"  Selftext (last 400 chars, where cut): ...{row['selftext'][-400:]}")

Documents truncated to 16000 chars. Rows affected: 10

Example truncated doc (post_id=t3_1bqhjgf, orig length=28224 chars):
  Total length now: 16000 chars
  Title: Just some help and support being kindly requested by a lethal cocktail of crippl...
  Selftext (last 400 chars, where cut): ...caught in the crosshairs of our dissensions, my parents constantly intervened and, reddit, please tell me if you think their conduct was right or not, but they would always tell me to take the high road because I was the elder brother and I should just let some of the insensitive comments she says be water under the bridge and that as the eldest I'm supposed to guide and mentor her. I started to i


In [8]:
from datasketch import MinHash, MinHashLSH

def get_minhash(text, num_perm=128):
    m = MinHash(num_perm=num_perm)
    for word in text.lower().split():
        m.update(word.encode("utf8"))
    return m

lsh = MinHashLSH(threshold=0.85, num_perm=128)
minhashes = {}

for _, row in clean_df.iterrows():
    pid = row["post_id"]
    mh = get_minhash(row["selftext"])
    minhashes[pid] = mh
    try:
        lsh.insert(pid, mh)
    except ValueError:
        pass  # duplicate minhash signature

near_dupes = []
for pid, mh in minhashes.items():
    results = lsh.query(mh)
    if len(results) > 1:
        near_dupes.append(results)

print(f"Near-duplicate clusters (>=85% similar): {len(near_dupes)}")

Near-duplicate clusters (>=85% similar): 5765


In [9]:
seen = set()
ids_to_drop = set()

for cluster in near_dupes:
    keep = cluster[0]
    for pid in cluster:
        if pid not in seen:
            seen.add(pid)
        else:
            continue
    for pid in cluster[1:]:
        ids_to_drop.add(pid)

before = len(clean_df)
clean_df = clean_df[~clean_df["post_id"].isin(ids_to_drop)]
after = len(clean_df)
print(f"Dropped {before - after} near-duplicate posts ({before} → {after})")

Dropped 3168 near-duplicate posts (138506 → 135338)


## Cost and Size Checks

In [10]:
import tiktoken

encoding = tiktoken.encoding_for_model("text-embedding-3-small")

def count_selftext_tokens(df, encoding):
    total_tokens = 0
    titles = df["title"].fillna("")
    selftexts = df["selftext"].fillna("")

    for i, (title, selftext) in enumerate(zip(titles, selftexts), 1):
        text = f"{title} | {selftext}"
        num_tokens = len(encoding.encode(str(text)))
        total_tokens += num_tokens
        if i % 20000 == 0:
            print(f"Token sum after {i} rows: {total_tokens:,}")

    print(f"Total tokens in df title|selftext: {total_tokens:,}")
    # openai text-embedding-3-small cost is $0.02 per million tokens
    cost = total_tokens / 1000000 * 0.02
    print(f"Cost: ${cost:.2f}")
    return

count_selftext_tokens(clean_df, encoding)


Token sum after 20000 rows: 6,991,929
Token sum after 40000 rows: 13,254,325
Token sum after 60000 rows: 19,676,188
Token sum after 80000 rows: 26,263,463
Token sum after 100000 rows: 33,009,009
Token sum after 120000 rows: 39,720,227
Total tokens in df title|selftext: 44,872,359
Cost: $0.90


In [11]:
BYTES_PER_VALUE = 4   # float32


def estimate_vector_store_size(num_vectors, dim, bytes_per_value=BYTES_PER_VALUE):
    total_bytes = num_vectors * dim * bytes_per_value
    total_mb = total_bytes / (1024 ** 2)
    total_gb = total_bytes / (1024 ** 3)
    print(f"Estimated raw embedding storage for {num_vectors:,} vectors:")
    print(f"~{total_mb:,.2f} MB (~{total_gb:,.2f} GB) of float32 embeddings")
    return


num_vectors = len(clean_df)
estimate_vector_store_size(num_vectors, 1536)
estimate_vector_store_size(num_vectors, 512)
estimate_vector_store_size(num_vectors, 256)

Estimated raw embedding storage for 135,338 vectors:
~793.00 MB (~0.77 GB) of float32 embeddings
Estimated raw embedding storage for 135,338 vectors:
~264.33 MB (~0.26 GB) of float32 embeddings
Estimated raw embedding storage for 135,338 vectors:
~132.17 MB (~0.13 GB) of float32 embeddings


## Supabase setup

In [11]:
import vecs
import os
import dotenv
dotenv.load_dotenv()
DB_CONNECTION = os.environ.get("DB_CONNECTION")

vx = vecs.create_client(DB_CONNECTION)

In [12]:
docs = vx.get_or_create_collection(name="mental_health", dimension=512)

In [13]:
from openai import OpenAI

dotenv.load_dotenv()

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
)


def create_embeddings(input):
    response = client.embeddings.create(
    model="text-embedding-3-small",
    input=input,
    encoding_format="float",
    dimensions=512
    )
    return response.data[0].embedding

In [ ]:
test_entry = clean_df.head(1)

len(response) = create_embeddings(test_entry)

## Upserting test

In [20]:
def create_embeddings_batch(texts):
    if not texts:
        return []
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts,
        encoding_format="float",
        dimensions=512
    )
    return [d.embedding for d in response.data]


def sanitize_metadata(row):
    """Vecs metadata: JSON-serializable primitives only (str, int, float, bool)."""
    return {
        "author": str(row["author"]) if pd.notna(row["author"]) else "",
        "created_utc": int(row["created_utc"]) if pd.notna(row["created_utc"]) else 0,
        "permalink": str(row["permalink"]) if pd.notna(row["permalink"]) else "",
        "subreddit": str(row["subreddit"]) if pd.notna(row["subreddit"]) else "",
        "subreddit_id": str(row["subreddit_id"]) if pd.notna(row["subreddit_id"]) else "",
        "link_flair_text": str(row["link_flair_text"]) if pd.notna(row["link_flair_text"]) else "",
        "retrieved_on": int(row["retrieved_on"]) if pd.notna(row["retrieved_on"]) else 0,
        "post_id": str(row["post_id"]),
        "title": str(row["title"]),
        "selftext": str(row["selftext"])
    }


BATCH_SIZE = 100
df_to_upsert = clean_df  # or clean_df

batch_df = df_to_upsert.head(BATCH_SIZE)
documents = (batch_df["title"] + " | " + batch_df["selftext"]).tolist()
embeddings = create_embeddings_batch(documents)

records = []
for i, (_, row) in enumerate(batch_df.iterrows()):
    post_id = str(row["post_id"])
    vec = embeddings[i]
    meta = sanitize_metadata(row)
    records.append((post_id, vec, meta))

docs.upsert(records=records)
print(f"Upserted {len(records)} records (ids = post_id).")

Upserted 100 records (ids = post_id).


In [ ]:
from sqlalchemy import text

with vx.engine.begin() as conn:
    conn.execute(text("SET LOCAL statement_timeout = '0'"))
    conn.execute(text("""
        CREATE INDEX IF NOT EXISTS ix_vec_cosine_hnsw
        ON vecs."mental_health"
        USING hnsw (vec vector_cosine_ops)
        WITH (m=16, ef_construction=64)
    """))
print("Index created successfully")

## Upsert to vecs with custom embeddings

Use `post_id` as the vector id. We create embeddings ourselves with OpenAI, then `docs.upsert(records=[(id, vector, metadata), ...])`.

In [25]:
# Full dataset: batch upsert with progress (resumable)
VECS_BATCH_SIZE = 100
vecs_progress_file = "data/vecs_upload_progress.txt"
df_to_upsert = clean_df

def load_vecs_progress(default_start: int = 0) -> int:
    if not os.path.exists(vecs_progress_file):
        return default_start
    with open(vecs_progress_file, "r") as f:
        return int(f.read().strip())


def save_vecs_progress(next_index: int) -> None:
    with open(vecs_progress_file, "w") as f:
        f.write(str(next_index))


start_idx = load_vecs_progress()
n_total = len(df_to_upsert)
print(f"Starting vecs upsert from index {start_idx} of {n_total}, batch size={VECS_BATCH_SIZE}.")

for batch_start in range(start_idx, n_total, VECS_BATCH_SIZE):
    batch_end = min(batch_start + VECS_BATCH_SIZE, n_total)
    batch_df = df_to_upsert.iloc[batch_start:batch_end]
    documents = (batch_df["title"] + " | " + batch_df["selftext"]).tolist()
    embeddings = create_embeddings_batch(documents)
    records = []
    for i, (_, row) in enumerate(batch_df.iterrows()):
        records.append((str(row["post_id"]), embeddings[i], sanitize_metadata(row)))
    try:
        docs.upsert(records=records)
        save_vecs_progress(batch_end)
        print(f"Upserted batch {batch_start}:{batch_end}.")
    except Exception as e:
        print(f"Error at batch {batch_start}:{batch_end}: {e}")
        break

Starting vecs upsert from index 0 of 135338, batch size=100.
Upserted batch 0:100.
Upserted batch 100:200.
Upserted batch 200:300.
Upserted batch 300:400.
Upserted batch 400:500.
Upserted batch 500:600.
Upserted batch 600:700.
Upserted batch 700:800.
Upserted batch 800:900.
Upserted batch 900:1000.
Upserted batch 1000:1100.
Upserted batch 1100:1200.
Upserted batch 1200:1300.
Upserted batch 1300:1400.
Upserted batch 1400:1500.
Upserted batch 1500:1600.
Upserted batch 1600:1700.
Upserted batch 1700:1800.
Upserted batch 1800:1900.
Upserted batch 1900:2000.
Upserted batch 2000:2100.
Upserted batch 2100:2200.
Upserted batch 2200:2300.
Upserted batch 2300:2400.
Upserted batch 2400:2500.
Upserted batch 2500:2600.
Upserted batch 2600:2700.
Upserted batch 2700:2800.
Upserted batch 2800:2900.
Upserted batch 2900:3000.
Upserted batch 3000:3100.
Upserted batch 3100:3200.
Upserted batch 3200:3300.
Upserted batch 3300:3400.
Upserted batch 3400:3500.
Upserted batch 3500:3600.
Upserted batch 3600:3700

In [14]:
post_query = """
Just a disclaimer, I am now 29 y/o F and this is something I wrote about, from when I was 16 and going through a pretty rough time. This is how I am trying to heal.

I would love to know what you guys think about it.

I do not have the confidence to get up and read this out loud, or to post this without anonymity, although I wish I did.

I was born into a house that should have been a sanctuary,

but my matriarch was a storm of violence and spirits.

A woman who used lies to kill a goodbye,

leaving behind two broken men

and a daughter who should not have been born.

I was a child who should have been saved by a system, a system that would have stopped the wind of her rage from blowing the roof off of my life.

My brother became a shadow behind a prison wall,

a long silence where a protector should have been.

And it left my dad, a loving father, but a poor parent.

He was built from sand, hollowing himself out with the sound of a can cracking open,

and using the needle to fill the void of grief that she had left behind.

That darkness was the only thing that could quiet his storm,

but it meant he loved me from a distance he couldn't cross.

He loved me, but he could not catch me.

So while I was still a girl, the world looked at the wreckage.

Trying to sort through the violence, the war, the cans and the tracks,

that system, old and broken signed a paper that said I was grown.

They sent me to a refuge that they told me was safe,

but the air inside was heavy.

It was just a place to put broken things.

Where the hallways were full of everyone's ghosts.

The loneliness that built a physical ache.

I was navigating a world of pain I was too young to understand.

I was numbing a terror with the same habits that I'd watched my whole life.

I was left starved for a connection.

I found people who knew the world longer than I had, and I used them for comfort, for a seat at the table.

I was looking for shelter while they used my innocence.

because I wished for safety and salvation from the pit I had been thrown.

I thought I had found my safety.

I thought I had found my salvation,

but I was thrown into a slavery that was framed as independence
"""

In [15]:
query_embeddings = create_embeddings(post_query)
documents = docs.query(
    data=query_embeddings,          
    limit=10,
    include_metadata=True,
)
for doc, metadata in documents:
    print(metadata["permalink"])

/r/mentalhealth/comments/1l10c58/anonymous_the_sacred_wasnt_safe_anymore/
/r/mentalhealth/comments/1ph5l1t/additional_flairs_sish_sa_violence_addiction/
/r/mentalhealth/comments/1lxfmdw/sometimes_i_feel_invisible_even_to_the_people_i/
/r/mentalhealth/comments/1ph3e1a/this_is_my_story/
/r/mentalhealth/comments/1p1sjqr/there_is_nothing_to_do_and_there_is_nowhere_to_go/
/r/mentalhealth/comments/1guriuk/letter_to_my_past_traumas/
/r/mentalhealth/comments/1ov4gmu/im_29_and_it_feels_like_my_whole_life_has_just/
/r/mentalhealth/comments/1leld9u/a_letter_ive_bin_working_on_to_send/
/r/mentalhealth/comments/1er64lm/something_i_needed_to_get_off_of_my_chest/
/r/mentalhealth/comments/1ez5fec/18m_dont_know_how_to_deal_with_my_childhood_trauma/


# Create ChromaDB

In [98]:
import chromadb
import os

client = chromadb.CloudClient(
  api_key=os.environ.get("CHROMA_API_KEY"),
)

In [ ]:
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

COLLECTION_NAME = "reddit-bot-vdb-2"

collection = client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=OpenAIEmbeddingFunction(
        model_name="text-embedding-3-small",
        api_key=os.environ.get("OPENAI_API_KEY"),
    ),
)

collection = client.get_collection(COLLECTION_NAME)

## Small batch test upload to chroma

In [70]:
chroma_test_df = clean_df.head(3)

In [102]:

documents = (
    chroma_test_df["title"]
    + " | "
    + chroma_test_df["selftext"]
).tolist()

ids = chroma_test_df["post_id"].astype(str).tolist()

metadatas = chroma_test_df[
    [
        "author",
        "created_utc",
        "permalink",
        "subreddit",
        "subreddit_id",
        "link_flair_text",
        "retrieved_on",
        "post_id",   
    ]
].to_dict(orient="records")

print(documents[:1])
print(ids[:3])
print(metadatas[:2])

["The Junkie, his ADHD, and his Moment of Clarity | I took to self-medicating at 14, long before my ADHD diagnosis at 23. By age 19, I had already been a chronic teenage alcoholic and graduated as a seasoned meth user by age 20. After years of inpatients, outpatients, slips, and relapses, I couldn't figure out what the hell was wrong with me. I tried the 12 steps, S.M.A.R.T. Recovery, health realization, etc... over and over, to no avail. I just couldn't stay sober and tried everything before I surrendered to the idea that complete abstinence just wasn't my thing. Keep in mind that I was still undiagnosed at this point.  \nHarm reduction is controversial. Some see it as swapping one bad habit out for another, while others advocate for it. I had no clue where to turn before my diagnosis, and I knew abstinence wasn't going to work. After some research, I finally found an alternative rehab that focused on co-occurring disorders as opposed to mere drug abuse. This place saved my fuckin' li

In [103]:
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
)

C:\Users\moust\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:08<00:00, 9.91MiB/s]


## Batch upload to safeguard against network errors/api timeout

In [96]:
clean_df.to_csv("data/clean_df.csv", index=False)

In [104]:
BATCH_SIZE = 300
progress_file = "data/chroma_upload_progress.txt"


def load_progress(default_start: int = 0) -> int:
    """Return the next row index to start from.

    If the progress file does not exist, start at `default_start`.
    If it exists, it should contain a single integer: the next index.
    """
    if not os.path.exists(progress_file):
        return default_start

    with open(progress_file, "r") as f:
        value = f.read().strip()
        return int(value)



def save_progress(next_index: int) -> None:
    with open(progress_file, "w") as f:
        f.write(str(next_index))


In [105]:
start = load_progress()
n = len(clean_df)

print(f"Starting upload from index {start} out of {n} rows, batch size={BATCH_SIZE}.")

for batch_start in range(start, n, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, n)
    print(f"Preparing batch {batch_start}:{batch_end} (size={batch_end - batch_start})")

    batch_df = clean_df.iloc[batch_start:batch_end]

    ids = batch_df["post_id"].astype(str).tolist()
    documents = (batch_df["title"] + " | " + batch_df["selftext"]).tolist()
    metadatas = batch_df[
        [
            "author",
            "created_utc",
            "permalink",
            "subreddit",
            "subreddit_id",
            "link_flair_text",
            "retrieved_on",
            "post_id",
        ]
    ].to_dict(orient="records")

    try:
        print(f"Uploading batch starting at index {batch_start} with {len(ids)} documents...")
        collection.add(ids=ids, documents=documents, metadatas=metadatas)
        print(f"Successfully uploaded batch {batch_start}:{batch_end}.")
    except Exception as e:
        print(f"Error uploading batch {batch_start}:{batch_end}: {e}")
        break

    save_progress(batch_end)
    print(f"Progress saved. Next start index will be {batch_end}.")


Starting upload from index 0 out of 143680 rows, batch size=300.
Preparing batch 0:300 (size=300)
Uploading batch starting at index 0 with 300 documents...
Successfully uploaded batch 0:300.
Progress saved. Next start index will be 300.
Preparing batch 300:600 (size=300)
Uploading batch starting at index 300 with 300 documents...
Successfully uploaded batch 300:600.
Progress saved. Next start index will be 600.
Preparing batch 600:900 (size=300)
Uploading batch starting at index 600 with 300 documents...
Successfully uploaded batch 600:900.
Progress saved. Next start index will be 900.
Preparing batch 900:1200 (size=300)
Uploading batch starting at index 900 with 300 documents...
Successfully uploaded batch 900:1200.
Progress saved. Next start index will be 1200.
Preparing batch 1200:1500 (size=300)
Uploading batch starting at index 1200 with 300 documents...
Successfully uploaded batch 1200:1500.
Progress saved. Next start index will be 1500.
Preparing batch 1500:1800 (size=300)
Uploa